# gm/ID results viewer  (no Spectre)

Reads a saved gm/ID lookup table (`*_gmid_lut.npz` produced by `device_char.ipynb`) and draws a **summary of plots** — ending with a **combo plot: gm/Id (left axis) + fT (right axis) vs Id/W**. No simulation is run; only NumPy + Matplotlib are needed.

## Step 0 — Preflight (Python + packages)

In [ ]:
import sys, os
MIN_PY = (3, 6)
_G,_R,_Z = chr(27)+'[1;42;30m', chr(27)+'[1;41;97m', chr(27)+'[0m'
def _pf(ok,msg): print((_G+' PASS '+_Z if ok else _R+' FAIL '+_Z)+'  '+msg); return ok
pyok=_pf(sys.version_info[:2]>=MIN_PY,'Python %d.%d.%d (need >= %d.%d)'%(sys.version_info[0],sys.version_info[1],sys.version_info[2],MIN_PY[0],MIN_PY[1]))
miss=[]
for m in ('numpy','matplotlib'):
    try: __import__(m); _pf(True,'package '+m)
    except Exception: miss.append(m); _pf(False,'package '+m+' (missing)')
if miss:
    import json,html
    from IPython.display import display,HTML
    cmd=sys.executable+' -m pip install --user '+' '.join(miss)
    oc='navigator.clipboard.writeText('+json.dumps(cmd)+");this.textContent='copied!'"
    print('Install the missing package(s):')
    display(HTML('<button onclick="'+oc.replace('"','&quot;')+'">copy</button> <code style="user-select:all">$ '+html.escape(cmd)+'</code>'))

## 1. Config — point at the lookup table

In [ ]:
%matplotlib inline
import numpy as np, json, html
import matplotlib.pyplot as plt
from IPython.display import display, HTML

LUT_PATH = 'sample_gmid_lut.npz'   # <-- the .npz to view (synthetic sample shipped in this repo)
W = 2e-6        # device width used during characterization (W_TOTAL); Id/W = current density [A/m]

def copy_cmd(cmd, note='run this in your own terminal'):
    oc='navigator.clipboard.writeText('+json.dumps(cmd)+");this.textContent='copied!';setTimeout(()=>this.textContent='copy',1500)"
    display(HTML('<div style="margin:6px 0;font-family:monospace;font-size:12px"><span style="color:#777">'
        +html.escape(note)+':</span><br><button onclick="'+oc.replace('"','&quot;')+'" style="cursor:pointer;'
        'border:1px solid #888;border-radius:4px;background:#eaeaea;padding:2px 10px;margin-right:8px">copy</button>'
        '<code style="user-select:all;background:#f5f5f5;border:1px solid #ddd;padding:3px 7px;border-radius:3px">$ '
        +html.escape(cmd)+'</code></div>'))

# the prefilled command: list what's inside the .npz
_peek = "import numpy as np; print(sorted(np.load('"+LUT_PATH+"').files)[:8])"
copy_cmd('python3 -c '+repr(_peek), 'peek at the lookup-table contents in your terminal')
print(sorted(np.load(LUT_PATH).files)[:8])

## 2. Load the table

In [ ]:
_G,_R,_Z = chr(27)+'[1;42;30m', chr(27)+'[1;41;97m', chr(27)+'[0m'
D = np.load(LUT_PATH)
devices = sorted({k.rsplit('__',1)[0] for k in D.files})   # device name may contain '__'
def grids(dev): return {k.rsplit('__',1)[1]: D[k] for k in D.files if k.rsplit('__',1)[0]==dev}
print((_G+' PASS '+_Z)+'  loaded '+os.path.basename(LUT_PATH)+'  ('+str(len(D.files))+' arrays)')
for dv in devices:
    g=grids(dv); print('   * %-28s L=%.3g..%.3g um, VDS 0..%.2f V, %d params'
          % (dv, g['L'][0], g['L'][-1], float(g['VDS'].max()), len([p for p in g if g[p].ndim==3])))

## 3. Summary: gm/Id vs current density (per L)

In [ ]:
def vds_half(g):
    vdd=float(g['VDS'].max()); return int(np.argmin(np.abs(g['VDS']-vdd/2))), vdd
n=len(devices)
fig,axs=plt.subplots(1,n,figsize=(6.5*n,4.6),squeeze=False); axs=axs.flat
for ax,dev in zip(axs,devices):
    g=grids(dev); iV,vdd=vds_half(g)
    for iL,L in enumerate(g['L']):
        Id=g['Id'][iL,iV,:]; msk=Id>0
        ax.semilogx((Id/W)[msk], g['gm_id'][iL,iV,:][msk], label='L=%.3g um'%L)
    ax.set(xlabel='Id/W  [A/m]', ylabel='gm/Id  [S/A]', title='%s  @ VDS=%.2f V'%(dev,vdd/2))
    ax.set_xlim(1e-4, 1e2)
    ax.grid(True,which='both',alpha=.3); ax.legend(fontsize=7)
fig.tight_layout()

## 4. Combo plot — gm/Id (left) + fT (right)  vs  Id/W

In [ ]:
n=len(devices)
fig,axs=plt.subplots(1,n,figsize=(7*n,5),squeeze=False); axs=axs.flat
for ax,dev in zip(axs,devices):
    g=grids(dev); iV,vdd=vds_half(g); iL=0          # Lmin
    Id=g['Id'][iL,iV,:]; msk=Id>0
    IdW=(Id/W)[msk]; gmid=g['gm_id'][iL,iV,:][msk]; ft=g['ft_GHz'][iL,iV,:][msk]
    ax2=ax.twinx()
    a,=ax.semilogx(IdW, gmid, 'b-',  lw=2, label='gm/Id')
    b,=ax2.loglog (IdW, ft,   'r--', lw=2, label='fT')      # fT on a LOG right axis
    ax.set_xlim(1e-4, 1e2)                                  # Id/W axis: 1e-4 .. 1e2 A/m
    ax.set_xlabel('Id/W  [A/m]')
    ax.set_ylabel('gm/Id  [S/A]', color='b'); ax2.set_ylabel('fT  [GHz] (log)', color='r')
    ax.tick_params(axis='y', colors='b'); ax2.tick_params(axis='y', colors='r')
    ax.set_title('%s   L=%.3g um,  VDS=%.2f V'%(dev, g['L'][iL], vdd/2))
    ax.grid(True, which='both', alpha=.3); ax.legend([a,b],['gm/Id','fT'], loc='upper center')
fig.suptitle('gm/Id (left axis) and fT (right axis)  vs  current density Id/W   @ Lmin, VDS=VDD/2')
fig.tight_layout()